<a href="https://colab.research.google.com/github/Jags-Kamani/ApacheSpark/blob/master/5_Department_Highest_Salary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question:

Write a solution to find employees who have the highest salary in each of the departments.
Return the result table in any order.

**Sample Input**

Employee Table


```
  id   name  salary  departmentId
0   1    Joe   70000             1
1   2    Jim   90000             1
2   3  Henry   80000             2
3   4    Sam   60000             2
4   5    Max   90000             1


```

Department Table



```
	id	Department
0	1	IT
1	2	Sales

```



**Sample output**

```
 Department Employee  Salary
0         IT      Jim   90000
1      Sales    Henry   80000
2         IT      Max   90000

```


**Your Answer here**

In [10]:
import pandas as pd

data1 = {
    'id' : [1, 2, 3, 4, 5],
    'name' : ['Joe', 'Jim', 'Henry', 'Sam', 'Max'],
    'salary' : [70000, 90000, 80000, 60000, 90000],
    'departmentId' : [1, 1, 2, 2, 1]
}

data2 = {
    'id' : [1, 2],
    'Department' : ['IT', 'Sales']
}

employee = pd.DataFrame(data1)
department = pd.DataFrame(data2)

#Solution using Pandas - transform:
#.transform('max') → assign max salary of that department to each row
#transform() keeps the same number of rows (unlike groupby().max())

def highestSalary(employee : pd.DataFrame, department : pd.DataFrame) -> pd.DataFrame:
  merged = employee.merge(department, left_on = 'departmentId', right_on = 'id', how = 'inner')
  max_salaries = merged.groupby('Department')['salary'].transform('max')
  highest_salary = merged[merged['salary'] == max_salaries][['Department', 'name', 'salary']]
  highest_salary.columns = ['department', 'employee', 'salary']
  return highest_salary

#Alternative Solution - Using dense rank in pandas:
def highestSalary(employee : pd.DataFrame, department : pd.DataFrame) -> pd.DataFrame:
  merged = employee.merge(department, left_on = 'departmentId', right_on= 'id', how='inner')
  merged['rank'] = merged.groupby('Department')['salary'].rank(method = 'dense', ascending = False)
  result = merged[merged['rank'] == 1][['Department', 'name', 'salary']]
  result.columns = ['department', 'employee', 'salary']
  return result

highestSalary(employee, department)


#PySpark Version:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank

spark = SparkSession.builder.appName("DepartmentHighestSalary").getOrCreate()

employee = spark.createDataFrame([
    (1, "Joe", 70000, 1),
    (2, "Jim", 90000, 1),
    (3, "Henry", 80000, 2),
    (4, "Sam", 60000, 2),
    (5, "Max", 90000, 1)
], ["id", "name", "salary", "departmentId"])


department = spark.createDataFrame([
    (1, "IT"),
    (2, "Sales")
], ["id", "name"])

merged = employee.join(department, employee.departmentId == department.id, "inner") \
                 .select(employee.name.alias( "employee"),
                         department.name.alias("department"),
                         employee.salary)

Window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

ranked_df = merged.withColumn("rank", dense_rank().over(Window_spec))

result = ranked_df.filter(col("rank") == 1) \
                  .select("department", "employee", "Salary")

result.show()


+----------+--------+------+
|department|employee|Salary|
+----------+--------+------+
|        IT|     Jim| 90000|
|        IT|     Max| 90000|
|     Sales|   Henry| 80000|
+----------+--------+------+



**Expected Output**



```
                  department employee  salary
2                   Marketing     Jane   99323
62                 Operations  Michael   99393
73           Customer Service     Jane   99963
79                Engineering  Michael   99055
83                    Finance     John   94868
87            Human Resources    Sarah   97143
88   Research and Development  Michael   98026
120                     Legal    Emily   99843
190                     Sales  Michael   94207
293                IT Support   Robert   98135
```

